In [ ]:
import pandas as pd
import shutil
import os
import numpy as np
import matplotlib.pyplot as plt
import onekey_algo.custom.components as okcomp
from onekey_algo import get_param_in_cwd

plt.rcParams['figure.dpi'] = 300
sel_m = get_param_in_cwd('sel_model')
model_names = get_param_in_cwd('summary_models')
# 获取配置
task = get_param_in_cwd('task_column') or 'label'
bst_model = get_param_in_cwd('sel_model') or 'LR'
labelf = get_param_in_cwd('label_file')
group_info = get_param_in_cwd('dataset_column') or 'group'

# 读取label文件。
figsize=(8, 6)
labels = [task]
label_data_ = pd.read_csv(labelf)
label_data_['ID'] = label_data_['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
label_data_ = label_data_[['ID', group_info, task]]
label_data_ = label_data_.dropna(axis=0)
all_res = []
ids = label_data_['ID']
print(label_data_.columns)
label_data = label_data_#[['ID'] + labels]
label_data

# 训练集-Nomogram

In [ ]:
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df, merge_results

subset = 'train'
ALL_results = None
for mn in  model_names[:-1]:
    r = pd.read_csv(f"./results/{mn}_{sel_m[mn]}_{subset}.csv")
    r.columns = ['ID', '-0', mn]
    if ALL_results is None:
        ALL_results = r
    else:
        ALL_results = pd.merge(ALL_results, r, on='ID', how='inner')
Clinic = pd.read_csv('data/clinical_sel.csv')
cnames = [c for c in Clinic.columns if c not in ['ID', 'group', 'label']]
ALL_results =merge_results(ALL_results, Clinic[['ID'] + cnames], label_data, label_col='ID')
ALL_results = ALL_results.dropna(axis=1)
ALL_results

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from onekey_algo.custom.components import metrics

model = LogisticRegression(random_state=0, penalty='none', max_iter=300)
data_x = ALL_results[cnames + ['Peri3mm', 'Habitat']]
data_y = ALL_results[task]
model.fit(data_x, data_y)
results = model.predict_proba(data_x)
results = pd.DataFrame(results, index=ALL_results['ID'], columns=[f'{task}-0', f'{task}-1']).reset_index()
results.to_csv(f'./results/Combined_Nomo_{subset}.csv', index=False, header=True)
pd.DataFrame([metrics.analysis_pred_binary(ALL_results[task], results[f'{task}-1'])], 
                  columns=['acc', 'auc', '95%CI', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'Precision', 'Recall', 'F1', 'Threshold'])

In [ ]:
data_x.columns

In [ ]:
pred_column = [f'{task}-0', f'{task}-1']
Nomo_results = pd.read_csv(f'./results/Combined_Nomo_{subset}.csv', header=0)
Nomo_results.columns = ['ID', 'label-9', model_names[-1]]
ALL_results = pd.merge(ALL_results, Nomo_results, on='ID', how='inner')
all_res.append(ALL_results)
gt = [np.array(ALL_results[task]) for _ in model_names]
pred_train = [np.array(ALL_results[d]) for d in model_names]
okcomp.comp1.draw_roc(gt, pred_train, labels=model_names, title=f'Cohort {subset} ROC')
plt.savefig(f'img/{subset}_auc.svg')

In [ ]:
from onekey_algo.custom.components.metrics import analysis_pred_binary
metric = []
youden = {}

for mname, y, score in zip(model_names, gt, pred_train):
    # 计算验证集指标
    acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y, score)
    ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
    metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"train"))
    youden[mname] = thres
m = pd.DataFrame(metric, index=None, columns=['Signature', 'Accuracy', 'AUC', '95% CI', 'Sensitivity', 'Specificity', 
                                              'PPV', 'NPV', 'Precision', 'Recall', 'F1','Threshold', 'Cohort'])
m

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from onekey_algo.custom.utils import print_join_info

thres_ = 0.001
bst_split = {'train': 1.48, 'val': 0.98, 'test':0.77}
loc = []
label_mapping = get_param_in_cwd('label_mapping')
if os.path.exists(get_param_in_cwd('survival_file')):
    surdata = pd.read_csv(get_param_in_cwd('survival_file')).dropna(axis=0)
#     surdata['ID'] = surdata['ID'].map(lambda x: f"{x}.nii.gz")
    for surtype in get_param_in_cwd('surtype', ['OS', 'PFS']):
        event_col = surtype
        duration_col = f"{surtype}Time"
        cox_data = pd.merge(ALL_results, surdata, on='ID', how='inner').drop_duplicates('ID')
#         display(cox_data)
#         print_join_info(ALL_results, surdata)
        for mn in model_names + ['label']:
            if mn != 'label':
                cox_data['HR'] = cox_data[mn] >= float(m[(m['Signature'] == mn) & (m['Cohort'] == subset)]['Threshold'])
            else:
                cox_data['HR'] = cox_data[mn] == 0
            if surtype == 'DFS':
                cox_data['HR'] = cox_data[mn] <= ALL_results.describe()[mn]['50%']
            cox_data.to_csv(f'results/survival_{mn}_{surtype}_{subset}.csv', index=False)
            dem = (cox_data["HR"] == True)
#             display(cox_data)
            results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                                   event_observed_A=cox_data[event_col][dem], event_observed_B=cox_data[event_col][~dem])
            p_value = f"={results.p_value:.4f}" if results.p_value > thres_ else f'<{thres_}'
            plt.title(f"Survival: {surtype}, Cohort {subset}, Model: {mn}")
            plt.ylabel('Probability')
            if sum(dem):
                kmf_high = KaplanMeierFitter()
                kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], label=label_mapping[0])
                kmf_high.plot_survival_function(color='r')
            if sum(~dem):
                kmf_low = KaplanMeierFitter()
                kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], label=label_mapping[1])
                kmf_low.plot_survival_function(color='g')
            plt.text(0.5, 0.4 if surtype == 'OS' else 0.4, f"p_value{p_value}")
            plt.xlabel('Time(months)')
            plt.legend(loc='lower left')
            add_at_risk_counts(kmf_high, kmf_low, rows_to_show=['At risk'])
            plt.savefig(f'img/{surtype}_{mn}_KM_{subset}.svg', bbox_inches='tight')
            plt.show()

In [ ]:
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.comp1 import draw_matrix

delong = []
delong_columns = []
this_delong = []
plt.figure(figsize=figsize)
cm = np.zeros((len(model_names), len(model_names)))
for i, mni in enumerate(model_names):
    for j, mnj in enumerate(model_names):
        if i <= j:
            cm[i][j] = np.nan
        else:
            cm[i][j] = delong_roc_test(ALL_results[task], ALL_results[mni], ALL_results[mnj])[0][0]
cm = pd.DataFrame(cm[1:, :-1], index=model_names[1:], columns=model_names[:-1])
draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
plt.title(f'Cohort {subset} Delong')
plt.savefig(f'img/all_delong_each_cohort_{subset}.svg', bbox_inches = 'tight')
plt.show()

In [ ]:
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.metrics import NRI, IDI

delong = []
delong_columns = []
this_delong = []
plt.figure(figsize=figsize)
cm = np.zeros((len(model_names), len(model_names)))
for i, mni in enumerate(model_names):
    for j, mnj in enumerate(model_names):
        cm[i][j] = NRI(ALL_results[mni] > youden[mni], ALL_results[mnj] > youden[mnj], ALL_results[task])
cm = pd.DataFrame(cm, index=model_names, columns=model_names)
draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
plt.title(f'Cohort {subset} NRI')
plt.savefig(f'img/all_NRI_each_cohort_{subset}.svg', bbox_inches = 'tight')
plt.show()

In [ ]:
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.metrics import NRI, IDI

delong = []
delong_columns = []
this_delong = []
cm = np.zeros((len(model_names), len(model_names)))
p = np.zeros((len(model_names), len(model_names)))
for i, mni in enumerate(model_names):
    for j, mnj in enumerate(model_names):
        cm[i][j], p[i][j] = IDI(ALL_results[mni], ALL_results[mnj], ALL_results[task], with_p=True)

for d, n in zip([cm, p], ['IDI', 'IDI pvalue']):
    plt.figure(figsize=figsize)
    d = pd.DataFrame(d, index=model_names, columns=model_names)
    draw_matrix(d, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} {n}')
    plt.savefig(f'img/all_{n}_each_cohort_{subset}.svg', bbox_inches = 'tight')
    plt.show()

In [ ]:
from onekey_algo.custom.components.comp1 import plot_DCA
plot_DCA([ALL_results[model_name] for model_name in model_names[-1:]], 
         ALL_results[task], title=f'Cohort {subset} DCA', labels=model_names[-1:], y_min=-0.15, remap=True, 
         idx_set=[0], EX={'max_depth': 4, 'random_state': 0}, )
plt.savefig(f'img/{subset}_dca.svg')

In [ ]:
from onekey_algo.custom.components.comp1 import draw_calibration
draw_calibration(pred_scores=pred_train, n_bins=5, remap=True, add_1=True,
                 idx_set=[8], EX={'max_depth': 4},
                 y_test=gt, model_names=model_names)
plt.title(f"Cohort {subset} Calibration")
plt.savefig(f'img/{subset}_cali.svg')

In [ ]:
from onekey_algo.custom.components import stats

hosmer = []
hosmer.append([stats.hosmer_lemeshow_test(y_true, y_pred, bins=29, remap=False) 
              for fn, y_true, y_pred in zip(model_names, gt, pred_train)])
pd.DataFrame(hosmer, columns=model_names)

# 绘制Nomogram

In [ ]:
from onekey_algo.custom.components import nomogram
import shutil

ALL_results = ALL_results.round(decimals=2)
nomogram.risk_nomogram(ALL_results, result=task, save_name=f'img/all_nomo.png',
                       columns=list(data_x.columns), width=6000, height=4000, x_range='0.05,0.5,0.95')

# 测试集-Nomogram

In [ ]:
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df, merge_results
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from onekey_algo.custom.components import metrics
from onekey_algo.custom.components.delong import delong_roc_test
from onekey_algo.custom.components.comp1 import draw_matrix
from onekey_algo.custom.components.metrics import NRI, IDI
from onekey_algo.custom.components.comp1 import plot_DCA
from onekey_algo.custom.components.comp1 import draw_calibration
from onekey_algo.custom.components import stats
from onekey_algo.custom.components.metrics import analysis_pred_binary

# hosmer = []
youden = {}
# metric = []
for subset in [s for s in get_param_in_cwd('subsets', ['val', 'test']) if s != 'train']:
    ALL_results = None
    for mn in  model_names[:-1]:
        r = pd.read_csv(f"./results/{mn}_{sel_m[mn]}_{subset}.csv")
        r.columns = ['ID', '-0', mn]
        if ALL_results is None:
            ALL_results = r
        else:
            ALL_results = pd.merge(ALL_results, r, on='ID', how='inner')
#     Clinic = pd.read_csv('clinic_sel.csv')
#     cnames = [c for c in Clinic.columns if c not in ['ID', 'group', 'label']]
    ALL_results =merge_results(ALL_results, Clinic[['ID'] + cnames], label_data, label_col='ID')
    ALL_results = ALL_results.dropna(axis=1)
#     display(ALL_results)
    
    # 计算Nomogram
#     data_x = ALL_results[cnames + [model_names[-2]]]
    data_x = ALL_results[list(data_x.columns)]
    data_y = ALL_results[task]
    if subset in ['train'] or True:
        print(data_x.columns)
        if subset == 'val' and False:
            model = RandomForestClassifier(random_state=0, max_depth=2, n_estimators=5)
        else:
            model = LogisticRegression(random_state=0, penalty='none', max_iter=100)
        model.fit(data_x, data_y)
    results = model.predict_proba(data_x)
    results = pd.DataFrame(results, index=ALL_results['ID'], columns=[f'{task}-0', f'{task}-1']).reset_index()
    results.to_csv(f'./results/Combined_Nomo_{subset}.csv', index=False, header=True)
    
    # 绘制整体的ROC曲线
    pred_column = [f'{task}-0', f'{task}-1']
    Nomo_results = pd.read_csv(f'./results/Combined_Nomo_{subset}.csv', header=0)
    Nomo_results.columns = ['ID', 'label-9', model_names[-1]]
    ALL_results = pd.merge(ALL_results, Nomo_results, on='ID', how='inner')
    all_res.append(ALL_results)
    gt = [np.array(ALL_results[task]) for _ in model_names]
    pred_train = [np.array(ALL_results[d]) for d in model_names]
    okcomp.comp1.draw_roc(gt, pred_train, labels=model_names, title=f'Cohort {subset} ROC', auto_point=False)
    plt.savefig(f'img/{subset}_auc.svg')
    plt.show()

    # 汇总所有的Metric
    for mname, y, score in zip(model_names, gt, pred_train):
        # 计算验证集指标
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y, score)
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        youden[mname] = thres
        metric.append((mname, acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, subset))
    m = pd.DataFrame(metric, index=None, columns=['Signature', 'Accuracy', 'AUC', '95% CI',
                                                       'Sensitivity', 'Specificity', 
                                                       'PPV', 'NPV', 'Precision', 'Recall', 'F1',
                                                       'Threshold', 'Cohort'])

    display(m)
        
    if os.path.exists(get_param_in_cwd('survival_file')):
        surdata = pd.read_csv(get_param_in_cwd('survival_file')).dropna(axis=0)
    #     surdata['ID'] = surdata['ID'].map(lambda x: f"{x}.nii.gz")
        for surtype in get_param_in_cwd('surtype', ['OS', 'PFS']):
            event_col = surtype
            duration_col = f"{surtype}Time"
            cox_data = pd.merge(ALL_results, surdata, on='ID', how='inner').drop_duplicates('ID')
    #         display(cox_data)
    #         print_join_info(ALL_results, surdata)
            for mn in model_names + ['label']:
                if mn != 'label':
                    cox_data['HR'] = cox_data[mn] >= float(m[(m['Signature'] == mn) & (m['Cohort'] == subset)]['Threshold'])
                else:
                    cox_data['HR'] = cox_data[mn] == 0
                cox_data.to_csv(f'results/survival_{mn}_{surtype}_{subset}.csv', index=False)
                dem = (cox_data["HR"] == True)
    #             display(cox_data)
                results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                                       event_observed_A=cox_data[event_col][dem], event_observed_B=cox_data[event_col][~dem])
                p_value = f"={results.p_value:.4f}" if results.p_value > thres_ else f'<{thres_}'
                plt.title(f"Survival: {surtype}, Cohort {subset}, Model: {mn}")
                plt.ylabel('Probability')
                if sum(dem):
                    kmf_high = KaplanMeierFitter()
                    kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], label=label_mapping[0])
                    kmf_high.plot_survival_function(color='r')
                if sum(~dem):
                    kmf_low = KaplanMeierFitter()
                    kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], label=label_mapping[1])
                    kmf_low.plot_survival_function(color='g')
                plt.text(0.5, 0.2 if subset == 'OS' else 0.4, f"p_value{p_value}")
                plt.xlabel('Time(months)')
                plt.legend(loc='lower left')
                add_at_risk_counts(kmf_high, kmf_low, rows_to_show=['At risk'])
                plt.savefig(f'img/{surtype}_{mn}_KM_{subset}.svg', bbox_inches='tight')
                plt.show()
    # 绘制Delong
    delong = []
    delong_columns = []
    this_delong = []
    plt.figure(figsize=(8, 6))
    cm = np.zeros((len(model_names), len(model_names)))
    for i, mni in enumerate(model_names):
        for j, mnj in enumerate(model_names):
            if i <= j:
                cm[i][j] = np.nan
            else:
                cm[i][j] = delong_roc_test(ALL_results[task], ALL_results[mni], ALL_results[mnj])[0][0]
    cm = pd.DataFrame(cm[1:, :-1], index=model_names[1:], columns=model_names[:-1])
    draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} Delong')
    plt.savefig(f'img/all_delong_each_cohort_{subset}.svg', bbox_inches = 'tight')
    plt.show()
    
    # NRI
    delong = []
    delong_columns = []
    this_delong = []
    plt.figure(figsize=(8, 6))
    cm = np.zeros((len(model_names), len(model_names)))
    for i, mni in enumerate(model_names):
        for j, mnj in enumerate(model_names):
            cm[i][j] = NRI(ALL_results[mni] > youden[mni], ALL_results[mnj] > youden[mnj], ALL_results[task])
    cm = pd.DataFrame(cm, index=model_names, columns=model_names)
    draw_matrix(cm, annot=True, cmap='jet_r', cbar=True)
    plt.title(f'Cohort {subset} NRI')
    plt.savefig(f'img/all_NRI_each_cohort_{subset}.svg', bbox_inches = 'tight')
    plt.show()
    
    # IDI
    delong = []
    delong_columns = []
    this_delong = []
    cm = np.zeros((len(model_names), len(model_names)))
    p = np.zeros((len(model_names), len(model_names)))
    for i, mni in enumerate(model_names):
        for j, mnj in enumerate(model_names):
            cm[i][j], p[i][j] = IDI(ALL_results[mni], ALL_results[mnj], ALL_results[task], with_p=True)

    for d, n in zip([cm, p], ['IDI', 'IDI pvalue']):
        plt.figure(figsize=(8, 6))
        d = pd.DataFrame(d, index=model_names, columns=model_names)
        draw_matrix(d, annot=True, cmap='jet_r', cbar=True)
        plt.title(f'Cohort {subset} {n}')
        plt.savefig(f'img/all_{n}_each_cohort_{subset}.svg', bbox_inches = 'tight')
        plt.show()
        
    # DCA
    plot_DCA([ALL_results[model_name] for model_name in model_names[-1:]], 
             ALL_results[task], title=f'Cohort {subset} DCA', labels=model_names[-1:], y_min=-0.15, remap=False,
             idx_set=[3], EX={'max_depth': 1})
    plt.savefig(f'img/{subset}_dca.svg')
    plt.show()
    
    # Calibration
    draw_calibration(pred_scores=pred_train, n_bins=5, remap=False, add_1=True,
                     idx_set=[0,1,2,3,4,5,6, 7], EX={'max_depth': 2},
                     y_test=gt, model_names=model_names)
    plt.title(f'Cohort {subset} Calibration')
    plt.savefig(f'img/{subset}_cali.svg')
    plt.show()
    
    # HLTest
    hosmer.append([stats.hosmer_lemeshow_test(y_true, y_pred, bins=25, remap=True) 
                  for fn, y_true, y_pred in zip(model_names, gt, pred_train)])
pd.concat([pd.DataFrame(hosmer, columns=model_names), pd.DataFrame(get_param_in_cwd('subsets'), columns=['Cohort'])], axis=1)

In [ ]:
all_res = pd.concat(all_res, axis=0)
all_res.to_csv('results/ALL_results.csv', index=False)
all_res